# Vaani turn-level ASR (Google Colab)

Transcribes speaker-safe turn clips exported by the local pipeline with
`ARTPARK-IISc/Vaani-FastConformer-Hindi` (NeMo).

**Inputs (upload to `/content`):**
- `<batch>_audio.zip` — the exported WAV clips
- `<batch>_manifest.json` — the turn manifest written next to the clips

**Outputs (downloaded):**
- `<batch>_asr_results.json` — JSON list of `{"audio_file": <basename>, "transcript": <raw text>}`
  consumed by `build_prompting_dialogue()`. Clips that failed to transcribe are
  **omitted** so they surface as `MISSING`; empty model output is kept as `""`
  so it surfaces as `EMPTY`.
- `<batch>_asr_report.json` — model, runtime, coverage and failures.

**Privacy:** clear all outputs before saving this notebook back into the
repository (Edit → Clear all outputs). Never commit transcripts, audio or tokens.
Transcripts are raw ASR text; do not edit or normalise them here.


In [ ]:
!pip install -q "nemo_toolkit[asr]" huggingface_hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from pathlib import Path

# Pending full-sample batch. For the earlier clean batch use
# prompting_turn_audio_clean.zip / prompting_turn_manifest_clean.json /
# prompting_turn_asr_results_clean.json.
ZIP_PATH = Path("/content/full_sample_pending_turn_audio.zip")
MANIFEST_PATH = Path("/content/full_sample_pending_turn_manifest.json")
OUTPUT_PATH = Path("/content/full_sample_pending_turn_asr_results.json")
REPORT_PATH = Path("/content/full_sample_pending_turn_asr_report.json")

EXTRACT_DIR = Path("/content") / ZIP_PATH.stem
MODEL_NAME = "ARTPARK-IISc/Vaani-FastConformer-Hindi"
BATCH_SIZE = 4

In [ ]:
import torch
import nemo
import nemo.collections.asr as nemo_asr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU is much faster.")

model = nemo_asr.models.ASRModel.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

print(f"Model: {MODEL_NAME}")
print(f"NeMo: {nemo.__version__}  Torch: {torch.__version__}  Device: {DEVICE}")

In [ ]:
import json
import re
import zipfile

from google.colab import files


def clip_name(value):
    # Manifests written on Windows store backslash paths; Path.name on Linux
    # would not split them.
    return re.split(r"[\\/]", str(value))[-1]


missing_uploads = [p.name for p in (ZIP_PATH, MANIFEST_PATH) if not p.exists()]

if missing_uploads:
    print(f"Upload: {missing_uploads}")
    files.upload()

for path in (ZIP_PATH, MANIFEST_PATH):
    if not path.exists():
        raise FileNotFoundError(f"{path.name} was not uploaded.")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(EXTRACT_DIR)

wav_by_name = {}

for path in sorted(EXTRACT_DIR.rglob("*.wav")):
    if path.name in wav_by_name:
        raise ValueError(f"Duplicate clip name in zip: {path.name}")
    wav_by_name[path.name] = path

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

if not isinstance(manifest, list):
    raise ValueError("Manifest must be a JSON list.")

expected = sorted(
    {
        clip_name(row["audio_file"])
        for row in manifest
        if row.get("export_for_asr") and row.get("audio_file")
    }
)

missing_clips = [name for name in expected if name not in wav_by_name]
unexpected_clips = sorted(set(wav_by_name) - set(expected))

print(f"Manifest rows:         {len(manifest)}")
print(f"Expected clips:        {len(expected)}")
print(f"WAV clips in zip:      {len(wav_by_name)}")
print(f"Missing from zip:      {len(missing_clips)}")
print(f"Not in manifest:       {len(unexpected_clips)}")

for name in missing_clips:
    print(f"  MISSING  {name}")

for name in unexpected_clips:
    print(f"  EXTRA    {name}")

# Only manifest clips are transcribed; extras are reported, not used.
to_transcribe = [wav_by_name[name] for name in expected if name in wav_by_name]

In [ ]:
def transcript_text(output):
    if isinstance(output, str):
        return output.strip()

    text = getattr(output, "text", None)

    if text is not None:
        return str(text).strip()

    return str(output).strip()


def transcribe_paths(paths):
    string_paths = [str(path) for path in paths]

    try:
        outputs = model.transcribe(
            audio=string_paths,
            batch_size=min(BATCH_SIZE, len(string_paths)),
        )
    except TypeError:
        outputs = model.transcribe(
            paths2audio_files=string_paths,
            batch_size=min(BATCH_SIZE, len(string_paths)),
        )

    # Some NeMo versions return (best_hypotheses, all_hypotheses).
    if isinstance(outputs, tuple):
        outputs = outputs[0]

    if len(outputs) != len(paths):
        raise RuntimeError(f"Expected {len(paths)} outputs, got {len(outputs)}.")

    return [transcript_text(output) for output in outputs]


results = []
failures = []

for start in range(0, len(to_transcribe), BATCH_SIZE):
    batch = to_transcribe[start : start + BATCH_SIZE]
    print(f"\nTranscribing {start + 1}-{start + len(batch)} / {len(to_transcribe)}")

    try:
        texts = transcribe_paths(batch)
        pairs = list(zip(batch, texts))
    except Exception as batch_error:
        print(f"  Batch failed ({batch_error!r}); retrying clips one by one.")
        pairs = []

        for path in batch:
            try:
                pairs.append((path, transcribe_paths([path])[0]))
            except Exception as clip_error:
                failures.append({"audio_file": path.name, "error": repr(clip_error)})
                print(f"  FAILED  {path.name}: {clip_error!r}")

    for path, text in pairs:
        results.append({"audio_file": path.name, "transcript": text})
        print(f"{path.name}\n  {text}")

In [ ]:
import datetime

empty = [row["audio_file"] for row in results if not row["transcript"]]

report = {
    "created_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model": MODEL_NAME,
    "nemo_version": nemo.__version__,
    "torch_version": torch.__version__,
    "device": DEVICE,
    "zip": ZIP_PATH.name,
    "manifest": MANIFEST_PATH.name,
    "results": OUTPUT_PATH.name,
    "expected_clips": len(expected),
    "transcribed": len(results),
    "empty_transcripts": empty,
    "failed": failures,
    "missing_from_zip": missing_clips,
    "not_in_manifest": unexpected_clips,
}

OUTPUT_PATH.write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

REPORT_PATH.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n=== ASR BATCH COMPLETE ===")
print(f"Expected:     {len(expected)}")
print(f"Transcribed:  {len(results)}")
print(f"Empty:        {len(empty)}")
print(f"Failed:       {len(failures)}")
print(f"Missing zip:  {len(missing_clips)}")

files.download(str(OUTPUT_PATH))
files.download(str(REPORT_PATH))